## Show multimodal data on tree

In [ ]:
import os
import sys
import dill
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")
sys.path.append(str(base_proj_dir))

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import plotting.settings
from adjustText import adjust_text
from dynaconf import Dynaconf
from tqdm import tqdm

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx


def plot_multi_lineage_trees_general(
    models,
    extractors,         
    gene,               
    tf,                 
    cmaps=None,
    nan_color="lightgrey",
    titles=None,
    fixed_vmin_vmax=None,
    figsize=(18, 5),
):

    n = len(models)

    if cmaps is None:
        cmaps = ["viridis"] * n

    if titles is None:
        titles = [f"Model {i+1}" for i in range(n)]

    fig, axes = plt.subplots(1, n, figsize=figsize, squeeze=False)
    axes = axes[0]

    for i, (model, extractor, cmap, title) in enumerate(zip(models, extractors, cmaps, titles)):

        ax = axes[i]

        G = model["graph"]
        pos = model["positions"]

        # Extract the values via custom extractor
        vals = extractor(model, gene, tf).reindex(G.nodes())

        # Handle NaNs
        nan_mask = vals.isna()
        numeric_vals = vals[~nan_mask]

        # Determine scaling
        if fixed_vmin_vmax is not None and fixed_vmin_vmax[i] is not None:
            vmin, vmax = fixed_vmin_vmax[i]
        else:
            vmin, vmax = numeric_vals.min(), numeric_vals.max()

        cmap_obj = plt.cm.get_cmap(cmap)
        norm = plt.Normalize(vmin=vmin, vmax=vmax)

        # Build per-node colors
        node_colors = []
        for node in G.nodes():
            v = vals.loc[node]
            if np.isnan(v):
                node_colors.append(nan_color)
            else:
                node_colors.append(cmap_obj(norm(v)))

        # Draw panel
        nx.draw(
            G,
            pos,
            node_color=node_colors,
            node_size=800,
            with_labels=True,
            font_size=9,
            ax=ax,
        )
        ax.set_title(title)

        # Add colorbar
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, shrink=0.8)
        cbar.set_label(title)

    plt.tight_layout()
    plt.show()

## Load data

In [ ]:
for target_tf in ["LMX1B"]:
    for target_gene in ["CHGA", "PAX4", "CACNA2D1", "KCNB2", "FEV"]:

        # Load VASA lineage model
        with open("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa_lineage_model.pkl", "rb") as f:
            gene_expression_lineage_model = dill.load(f)
            
        # Load Perturb lineage model
        with open(Path(settings.ANALYSIS.models_dir) / f"parse_{target_tf.lower().capitalize()}_perturb_lineage_model.pkl", "rb") as f:
            perturb_lineage_model = dill.load(f)
            
        # Load multiome lineage model
        with open("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/grn/multiome_lineage_model.pkl", "rb") as f:
            tf_accessibility_lineage_model = dill.load(f)

        def extract_expression(model, gene, tf):
            return model["mean_expression"][gene]

        def extract_perturbation(model, gene, tf):
            return model["mean_coef"][gene]

        def extract_tf_accessibility(model, gene, tf):
            return model["mean_accessibility"][tf]

        models = [
            gene_expression_lineage_model,   # baseline RNA
            perturb_lineage_model,           # perturbed model
            tf_accessibility_lineage_model,  # accessibility model
        ]

        extractors = [
            extract_expression,
            extract_perturbation,
            extract_tf_accessibility,
        ]

        titles = [
            f"{target_gene} Expression",
            f"{target_tf} Perturbation Effect on {target_gene}",
            f"{target_tf} Accessibility",
        ]

        cmaps = ["RdPu", "RdBu_r", "BuGn"]

        fixed_vmin_vmax = [
                None,          # auto scale for expression
            (-2, 2),       # fixed scale for perturbation coefficients
            None
        ]

        plot_multi_lineage_trees_general(
            models=models,
            extractors=extractors,
            gene=target_gene,
            tf=target_tf,
            cmaps=cmaps,
            nan_color="lightgrey",
            titles=titles,
            fixed_vmin_vmax=fixed_vmin_vmax,
        )

        plt.savefig(Path(settings.ANALYSIS.figures_dir) / f"{target_tf}_{target_gene}_multi_modal_lineage_trees.pdf")